# Exercise 11 — Segmentation: which pixels are ink?

## What you are about to see

Three exercises, three sizes of answer:

| exercise | question | answer per image |
|---|---|---|
| [09](q09_image_classification.ipynb) classification | which digit is this? | **one** label |
| [10](q10_object_detection.ipynb) detection | which digits, and where? | **one label and one box per object** |
| **11 segmentation** | what is each pixel? | **one label per pixel** — an image the size of the input |

The data are ten pages of old handwritten documents, from the H-DIBCO 2018 competition
(Pratikakis et al., 2018). Each pixel is either **ink** or **background** — and the
background is not clean paper: it has stains, uneven light, and ink that soaked through
from the other side of the sheet. Separating the writing from all of that is called
**binarisation**, and it is the first step before any text recognition.

The network is an **encoder–decoder**: convolutions with stride 2 shrink the image into
a small map of features, then **transposed convolutions** (`ConvTranspose2d`) grow it back
to the full size, so the output has one value per input pixel.

In [ ]:
# Run me first: make ``dlcb`` and ``datasets`` importable whether Jupyter was
# started in the project root or inside ``exercises/``.
import pathlib
import sys

ROOT = pathlib.Path.cwd()
if not (ROOT / "dlcb").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import time
from typing import List, Tuple

import matplotlib.pyplot as plt
import numpy as np

import datasets
from dlcb import engine as cpu
from dlcb import nn, optim
from dlcb.conv import Conv2d, ConvTranspose2d

print("ready — project imported from", ROOT)

## 1. Input and output

`datasets.load_dibco2018()` returns ten pages. For page $i$:

| | what | shape | values |
|---|---|---|---|
| **input** | the scan, in colour | $(H_i, W_i, 3)$ | RGB in $[0, 1]$ |
| **output** (the target) | the ink mask | $(H_i, W_i)$ | **1 = ink**, 0 = background |

Input and output have **the same height and width**, pixel for pixel: `mask[r, c]` says
whether pixel `image[r, c]` is ink. And every page has its own size — from about
$300 \times 1200$ to $900 \times 3900$ pixels — so the pages are a list, not one array.

In [ ]:
dibco = datasets.load_dibco2018()
print(dibco)
print(f"\n{'page':>4} {'input shape':>17} {'target shape':>14} {'ink pixels':>11}")
for i in range(len(dibco)):
    image, mask = dibco[i]
    print(f"{dibco.names[i]:>4} {str(image.shape):>17} {str(mask.shape):>14} {mask.mean():>11.1%}")

One page, its target, and a close-up of both. Look at the close-up: the faint, mirrored
writing that shows through from the back of the sheet is **background** in the target —
it is dark, but it is not the ink of this page. A rule like "dark pixel = ink" cannot get
that right.

In [ ]:
def show_pair(image: np.ndarray, mask: np.ndarray, title: str, rows=None, cols=None) -> None:
    """The input next to its target, optionally a crop of both."""
    rows = rows or slice(None)
    cols = cols or slice(None)
    fig, (a, b) = plt.subplots(1, 2, figsize=(12, 3.2))
    a.imshow(image[rows, cols], cmap="gray" if image.shape[-1] == 1 else None, vmin=0, vmax=1)
    a.set_title(f"INPUT: {title}", fontsize=10)
    b.imshow(mask[rows, cols], cmap="gray_r", vmin=0, vmax=1)
    b.set_title("TARGET: 1 = ink (black), 0 = background (white)", fontsize=10)
    for ax in (a, b):
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()


image, mask = dibco[0]
show_pair(image, mask, f"page {dibco.names[0]}, {image.shape[0]} × {image.shape[1]}")
show_pair(image, mask, "close-up", slice(80, 330), slice(150, 900))

**Ink is rare.** It covers between 3 % and 16 % of a page. So a "network" that answers
*background* for every pixel is right about 92 % of the time — and has found no ink at
all. Pixel accuracy is the wrong score for this task; Section 7 uses scores that look at
the ink.

In [ ]:
total = sum(m.size for m in dibco.masks)
ink = sum(m.sum() for m in dibco.masks)
print(f"all pages: {total:,} pixels, {ink:,} of them ink ({ink / total:.1%})")
print(f"answering 'background' everywhere: pixel accuracy {1 - ink / total:.1%}, ink found 0 %")

## 2. Preparing the input

Three decisions, each made for a reason you can check:

1. **Grey levels.** Ink and paper differ mainly in brightness, so the network gets one
   channel instead of three — a third of the arithmetic in the first layer.
2. **Each page standardised on its own**: subtract the page's median grey level, divide by
   its standard deviation. Pages were scanned under different light; one has bright paper
   and dark ink, another has grey paper and ink only a little darker. After this step
   "darker than this page's paper" means the same number on every page.
3. **Train on some pages, test on others.** Seven pages for training, three held out. A
   page shares its writer, its paper and its stains with every crop cut from it, so
   testing on crops of a training page would measure memory, not generalisation.

The network works in **`float32`** (`cpu.default_dtype`): half the memory of the default
`float64`, and enough precision for training. Every tensor created after this line uses it.

In [ ]:
cpu.default_dtype = np.float32

TEST_PAGES = [1, 4, 9]                                   # pages "2", "5" and "10"
TRAIN_PAGES = [i for i in range(len(dibco)) if i not in TEST_PAGES]


def prepare(i: int) -> np.ndarray:
    # Page i as the network sees it: grey, standardised on its own, (H, W, 1) float32.
    grey = dibco.grey(i)
    return ((grey - np.median(grey)) / grey.std()).astype(np.float32)


pages = [prepare(i) for i in range(len(dibco))]
print("train pages:", [dibco.names[i] for i in TRAIN_PAGES],
      f"({sum(dibco.masks[i].size for i in TRAIN_PAGES):,} pixels)")
print("test pages: ", [dibco.names[i] for i in TEST_PAGES],
      f"({sum(dibco.masks[i].size for i in TEST_PAGES):,} pixels)")
print(f"\npage 1 after preparing: shape {pages[0].shape}, dtype {pages[0].dtype}, "
      f"median {np.median(pages[0]):.2f}, std {pages[0].std():.2f}")

### Training on crops

A whole page is too large for one training step, and the pages have different sizes. So
each step trains on a **batch of $64 \times 64$ crops**, cut at random places of random
training pages (a larger page is picked more often, in proportion to its area). Every
step sees new crops: over a few hundred steps the network meets every part of every
training page, never twice in exactly the same frame.

In [ ]:
CROP = 64
BATCH = 16
rng = np.random.RandomState(0)
area = np.array([dibco.masks[i].size for i in TRAIN_PAGES], dtype=float)


def random_crops(n: int) -> Tuple[np.ndarray, np.ndarray]:
    """n random CROP × CROP crops of training pages: inputs (n, 64, 64, 1), targets (n, 64, 64, 1)."""
    xs, ys = [], []
    for i in rng.choice(TRAIN_PAGES, size=n, p=area / area.sum()):
        h, w = dibco.masks[i].shape
        top, left = rng.randint(0, h - CROP + 1), rng.randint(0, w - CROP + 1)
        xs.append(pages[i][top:top + CROP, left:left + CROP])
        ys.append(dibco.masks[i][top:top + CROP, left:left + CROP, None])
    return np.stack(xs), np.stack(ys).astype(np.float32)


x, y = random_crops(8)
print(f"one batch: inputs {x.shape} {x.dtype}, targets {y.shape} {y.dtype}")
fig, axes = plt.subplots(2, 8, figsize=(13, 3.6))
for k in range(8):
    axes[0, k].imshow(x[k, :, :, 0], cmap="gray")
    axes[1, k].imshow(y[k, :, :, 0], cmap="gray_r", vmin=0, vmax=1)
    axes[0, k].set_title(f"crop {k}", fontsize=9)
for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])
axes[0, 0].set_ylabel("input"); axes[1, 0].set_ylabel("target")
plt.tight_layout()
plt.show()

## 3. The network: down, up, and skip connections

The output must be as large as the input. Two simple designs fail:

- **Convolutions with stride 1 only** keep the size, but each output pixel then sees only
  a small window — a few $3 \times 3$ layers see $7 \times 7$ or $9 \times 9$ pixels. Not
  enough to tell a stroke from a stain of the same darkness: that needs context.
- **Shrinking, as in Exercise 09**, gives context — every stride-2 layer doubles how far
  a feature sees — but the output is then smaller than the input.

The **encoder–decoder** does both. The *encoder* shrinks: $64 \to 32 \to 16$, with more
channels at each level. The *decoder* grows back: $16 \to 32 \to 64$, with
`ConvTranspose2d(…, kernel 4, stride 2, padding 1)`, which exactly doubles the height and
width, $(H - 1) \cdot 2 - 2 + 4 = 2H$.

Growing back from $16 \times 16$ cannot restore where a stroke's edge was to the pixel —
that detail was averaged away on the way down. So the decoder also receives, at each
level, the encoder's features **of the same size**, concatenated along the channel axis
(`cpu.cat(…, axis=3)`). These **skip connections** carry the fine detail across; the deep
path carries the context. This is the design of U-Net (Ronneberger, Fischer & Brox,
2015), here with two levels instead of four:

```
input 64×64×1 ─ conv 3×3 ─────────── a1: 64×64×8 ──────────────────────────┐ skip
                  │ conv 3×3, stride 2                                        │
                  ▼                                                           │
               32×32×16 ─ conv 3×3 ─ a2: 32×32×16 ──────────┐ skip            │
                             │ conv 3×3, stride 2           │                 │
                             ▼                              │                 │
                          16×16×32 ─ conv 3×3 ─ b: 16×16×32 │                 │
                                                  │ ConvTranspose2d, ×2       │
                                                  ▼         ▼                 │
                                          32×32×16  ⧺  a2 → conv 3×3 → 32×32×16
                                                                │ ConvTranspose2d, ×2
                                                                ▼             ▼
                                                        64×64×8  ⧺  a1 → 64×64×16
                                                                          │ conv 1×1
                                                                          ▼
                                                              output 64×64×1: one logit per pixel
```

The last layer is a $1 \times 1$ convolution — a dense layer applied to every pixel
separately, as in Exercise 10 — that turns the 16 features of a pixel into one number,
the **logit** $z$; $\sigma(z)$ is the probability that the pixel is ink.

The network has **no dense layer over the whole image**, only convolutions. So its
weights do not depend on the image size, and the same network trained on $64 \times 64$
crops runs unchanged on a $922 \times 3933$ page: a *fully convolutional network* (Long,
Shelhamer & Darrell, 2015).

In [ ]:
class InkNet(nn.Module):
    """A two-level U-Net: (batch, H, W, 1) -> (batch, H, W, 1) logits, for H, W multiples of 4."""

    def __init__(self, width: int = 8) -> None:
        w = width
        self.enc1 = Conv2d(1, w, 3, padding=1)                     # 64×64:  1 -> 8
        self.down1 = Conv2d(w, 2 * w, 3, stride=2, padding=1)      # 64 -> 32
        self.enc2 = Conv2d(2 * w, 2 * w, 3, padding=1)             # 32×32: 16 -> 16
        self.down2 = Conv2d(2 * w, 4 * w, 3, stride=2, padding=1)  # 32 -> 16
        self.bottom = Conv2d(4 * w, 4 * w, 3, padding=1)           # 16×16: 32 -> 32
        self.up2 = ConvTranspose2d(4 * w, 2 * w, 4, stride=2, padding=1)   # 16 -> 32
        self.dec2 = Conv2d(4 * w, 2 * w, 3, padding=1)             # 32×32: (16 + 16 skip) -> 16
        self.up1 = ConvTranspose2d(2 * w, w, 4, stride=2, padding=1)       # 32 -> 64
        self.head = Conv2d(2 * w, 1, 1)                            # 64×64: (8 + 8 skip) -> 1

    def forward(self, x: cpu.Tensor) -> cpu.Tensor:
        a1 = self.enc1(x).relu()
        a2 = self.enc2(self.down1(a1).relu()).relu()
        b = self.bottom(self.down2(a2).relu()).relu()
        d2 = self.dec2(cpu.cat([self.up2(b).relu(), a2], axis=3)).relu()    # skip from a2
        d1 = cpu.cat([self.up1(d2).relu(), a1], axis=3)                      # skip from a1
        return self.head(d1)


cpu.set_seed(0)
model = InkNet()
print(f"parameters: {sum(p.data.size for p in model.parameters()):,}")
with cpu.no_grad():
    out = model(cpu.Tensor(x))
print(f"input {x.shape} -> output {out.shape}: one logit per input pixel")

## 4. The loss: one yes/no question per pixel

Every pixel asks the same yes/no question, so the loss is the binary cross-entropy of
Exercise 10's objectness, averaged over all pixels of the batch:

$$
\mathcal{L} = \frac{1}{N} \sum_{\text{pixels}} \Bigl[ -y \log \sigma(z) - (1 - y) \log \bigl(1 - \sigma(z)\bigr) \Bigr],
$$

with $y \in \{0, 1\}$ the target and $z$ the logit; computed from $z$ directly in the
stable form $\max(z, 0) - z\,y + \log(1 + e^{-|z|})$.

In [ ]:
def pixel_bce(z: cpu.Tensor, y: np.ndarray) -> cpu.Tensor:
    """Mean binary cross-entropy of σ(z) against 0/1 targets, one term per pixel."""
    t = cpu.Tensor(y, requires_grad=False)
    abs_z = z.relu() + (-z).relu()
    return (z.relu() - z * t + ((-abs_z).exp() + 1.0).log()).mean()


with cpu.no_grad():
    print(f"untrained loss on one batch: {float(pixel_bce(model(cpu.Tensor(x)), y).data):.3f}"
          f"   (ln 2 = {np.log(2):.3f}: a coin flip per pixel)")

## 5. Training

700 steps of `BATCH` random crops, Adam. It takes about **six and a half minutes**: every convolution
runs at every pixel, and the first and last layers work at the full $64 \times 64$.

In [ ]:
STEPS = 700
LR = 2e-3

opt = optim.Adam(model.parameters(), lr=LR)
started = time.time()
losses = []
for step in range(1, STEPS + 1):
    x, y = random_crops(BATCH)
    opt.zero_grad()
    loss = pixel_bce(model(cpu.Tensor(x)), y)
    loss.backward()
    opt.step()
    losses.append(float(loss.data))
    if step % 100 == 0:
        print(f"step {step:4d}   loss {np.mean(losses[-100:]):.4f}   [{time.time() - started:4.0f} s]")

## 6. Segmenting a whole page

The network is fully convolutional, so it could take a whole page at once — but the
intermediate maps of a $922 \times 3933$ page would need gigabytes. So the page is cut
into $256 \times 256$ **tiles**, each tile is segmented, and the results are put back in
place. The page is first padded on the right and bottom, by repeating its last row and
column, to a multiple of 256; the padding is cut off at the end.

The output is a **probability map** $p = \sigma(z)$, one value per pixel; the
**binarised page** is $p > 0.5$.

In [ ]:
TILE = 256


def ink_probability(page: np.ndarray) -> np.ndarray:
    """A prepared page (H, W, 1) -> probability of ink for every pixel, (H, W)."""
    h, w, _ = page.shape
    hp, wp = -(-h // TILE) * TILE, -(-w // TILE) * TILE           # round up to the tile
    padded = np.pad(page, ((0, hp - h), (0, wp - w), (0, 0)), mode="edge")
    logits = np.zeros((hp, wp), dtype=np.float32)
    with cpu.no_grad():
        for top in range(0, hp, TILE):
            for left in range(0, wp, TILE):
                tile = padded[None, top:top + TILE, left:left + TILE]
                logits[top:top + TILE, left:left + TILE] = model(cpu.Tensor(tile)).data[0, :, :, 0]
    return 1.0 / (1.0 + np.exp(-logits[:h, :w].astype(np.float64)))


started = time.time()
prob = {i: ink_probability(pages[i]) for i in range(len(dibco))}
print(f"all ten pages segmented in {time.time() - started:.0f} s")

A test page — one the network never trained on: the input, the probability map, and the
binarised result next to the target.

In [ ]:
def show_result(i: int, rows=slice(None), cols=slice(None)) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(13, 5.2))
    axes[0, 0].imshow(dibco.images[i][rows, cols])
    axes[0, 0].set_title(f"input (page {dibco.names[i]})", fontsize=10)
    axes[0, 1].imshow(prob[i][rows, cols], cmap="gray_r", vmin=0, vmax=1)
    axes[0, 1].set_title("probability of ink, σ(z)", fontsize=10)
    axes[1, 0].imshow(prob[i][rows, cols] > 0.5, cmap="gray_r", vmin=0, vmax=1)
    axes[1, 0].set_title("OUTPUT: binarised, p > 0.5", fontsize=10)
    axes[1, 1].imshow(dibco.masks[i][rows, cols], cmap="gray_r", vmin=0, vmax=1)
    axes[1, 1].set_title("TARGET", fontsize=10)
    for ax in axes.flat:
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()


show_result(TEST_PAGES[0])

## 7. How good is it?

Count, over the pixels of a page, the four outcomes of a yes/no prediction, with **ink as
the positive class**:

| | target ink | target background |
|---|---|---|
| **predicted ink** | TP (true positive) | FP (false positive) |
| **predicted background** | FN (false negative) | TN (true negative) |

and from them the scores the DIBCO competitions report (Pratikakis et al., 2018):

- **precision** $P = \frac{TP}{TP + FP}$ — of the pixels called ink, how many are ink;
- **recall** $R = \frac{TP}{TP + FN}$ — of the ink, how much was found;
- **F-measure** $F = \frac{2PR}{P + R}$ — their harmonic mean, high only if both are;
- **PSNR** $= 10 \log_{10} \frac{1}{\mathrm{MSE}}$ dB, with MSE the fraction of wrong
  pixels — how close the binarised page is to the target as an image; higher is better.

Pixel accuracy is printed too, next to the accuracy of answering "background" everywhere,
to show why it is not used.

In [ ]:
def scores(pred: np.ndarray, target: np.ndarray) -> dict:
    """Precision, recall, F-measure, PSNR and accuracy of a binary prediction, ink = positive."""
    tp = np.sum(pred & target); fp = np.sum(pred & ~target); fn = np.sum(~pred & target)
    precision, recall = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    wrong = (fp + fn) / target.size
    return {"P": precision, "R": recall, "F": 2 * precision * recall / max(precision + recall, 1e-12),
            "PSNR": 10 * np.log10(1 / wrong), "acc": 1 - wrong, "acc0": 1 - target.mean()}


print(f"{'page':>5} {'split':>6} {'precision':>10} {'recall':>7} {'F':>6} {'PSNR':>6} "
      f"{'accuracy':>9} {'all-bg acc':>11}")
results = {}
for i in TRAIN_PAGES + TEST_PAGES:
    results[i] = s = scores(prob[i] > 0.5, dibco.masks[i] == 1)
    split = "test" if i in TEST_PAGES else "train"
    print(f"{dibco.names[i]:>5} {split:>6} {s['P']:>10.3f} {s['R']:>7.3f} {s['F']:>6.3f} "
          f"{s['PSNR']:>6.1f} {s['acc']:>9.3f} {s['acc0']:>11.3f}")
for split, idx in (("train", TRAIN_PAGES), ("test", TEST_PAGES)):
    print(f"mean F, {split} pages: {np.mean([results[i]['F'] for i in idx]):.3f}")

- **Accuracy says nothing.** Every page scores 94–99 %, but answering "background"
  everywhere already scores 85–97 %.
- **F-measure: 0.89 on training pages, 0.72 on test pages.** Seven pages are little
  data; the gap is the cost of a new writer, new paper and new stains.
- **Page 10 is the easiest test page (0.87)** although its ink is faint: per-page
  standardisation makes "lighter than this page's paper" the same number everywhere.

## 8. Error analysis

### Where are the wrong pixels?

Colour every pixel by its outcome: **black** = ink found (TP), **red** = called ink but
background (FP), **blue** = ink missed (FN), white = background correctly left out (TN).
Below, for each test page, the $200 \times 700$ window with the most ink.

In [ ]:
def outcome_image(pred: np.ndarray, target: np.ndarray) -> np.ndarray:
    """RGB image: TP black, FP red, FN blue, TN white."""
    rgb = np.ones(target.shape + (3,))
    rgb[pred & target] = (0, 0, 0)
    rgb[pred & ~target] = (0.9, 0.1, 0.1)
    rgb[~pred & target] = (0.1, 0.3, 0.95)
    return rgb


def densest_window(mask: np.ndarray, height: int = 200, width: int = 700):
    # The height × width window (on a 50-pixel grid) that holds the most target ink.
    best, where = -1, (0, 0)
    for top in range(0, mask.shape[0] - height + 1, 50):
        for left in range(0, mask.shape[1] - width + 1, 50):
            ink = mask[top:top + height, left:left + width].sum()
            if ink > best:
                best, where = ink, (top, left)
    return slice(where[0], where[0] + height), slice(where[1], where[1] + width)


CLOSE_UPS = {i: densest_window(dibco.masks[i]) for i in TEST_PAGES}
fig, axes = plt.subplots(len(TEST_PAGES), 2, figsize=(13, 2.6 * len(TEST_PAGES)))
for row, i in enumerate(TEST_PAGES):
    r, c = CLOSE_UPS[i]
    axes[row, 0].imshow(dibco.images[i][r, c])
    axes[row, 0].set_title(f"page {dibco.names[i]}: input", fontsize=10)
    axes[row, 1].imshow(outcome_image(prob[i][r, c] > 0.5, dibco.masks[i][r, c] == 1))
    axes[row, 1].set_title("black = found, red = false ink, blue = missed ink", fontsize=10)
for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

### Edges or whole strokes?

A wrong pixel can be wrong in two very different ways:

- **on the edge of a stroke** — the stroke is there, but one pixel too thick or too thin.
  Where exactly a blurred stroke ends is uncertain even for the person who drew the
  target, so these errors are mostly harmless;
- **far from any edge** — a whole stroke missed, or a stain or a show-through letter
  called ink. These are the errors that change what the page says.

To separate them, mark the **edge band**: the pixels within one pixel of the boundary
between ink and background in the target. In code, a pixel is in the band if its
$3 \times 3$ neighbourhood holds both ink and background.

In [ ]:
def edge_band(target: np.ndarray) -> np.ndarray:
    """Pixels whose 3×3 neighbourhood (in the target) contains both ink and background."""
    padded = np.pad(target, 1, mode="edge")
    h, w = target.shape
    shifts = [padded[dr:dr + h, dc:dc + w] for dr in range(3) for dc in range(3)]
    return np.any(shifts, axis=0) & ~np.all(shifts, axis=0)


print(f"{'page':>5} {'wrong px':>9} {'on edges':>9} {'FP far':>8} {'FN far':>8}   band = % of page")
for i in TEST_PAGES:
    target, pred = dibco.masks[i] == 1, prob[i] > 0.5
    band = edge_band(target)
    wrong = pred != target
    print(f"{dibco.names[i]:>5} {wrong.sum():>9,} {np.mean(band[wrong]):>9.1%} "
          f"{np.sum(wrong & ~band & pred):>8,} {np.sum(wrong & ~band & ~pred):>8,}"
          f"   {band.mean():.1%}")

Only 20–30 % of the wrong pixels sit on stroke edges. Most errors are far from any
edge, so they are *structural*:

- **page 2** — false ink (28 621 px): the dark, torn border of the sheet looks like ink to
  a network that never saw a border in training (see the figure of Section 6);
- **page 5** — missed ink (19 344 px): a second, much fainter hand, visible as blue lines in
  the close-up;
- **page 10** — isolated red specks on stains, and short blue gaps inside faint strokes.

### The threshold

The binarised page is $p > \tau$ with $\tau = 0.5$. A lower $\tau$ calls more pixels ink:
recall rises, precision falls. The network was never told about $\tau$, so it is fair to
ask which value would have been best — on the **training** pages, and then to check it on
the test pages; choosing it on the test pages would make the test score optimistic.

In [ ]:
taus = np.round(np.arange(0.1, 0.91, 0.1), 1)
print(f"{'tau':>5} {'train F':>8} {'test F':>7} {'test P':>7} {'test R':>7}")
best_tau, best_f = None, -1.0
for tau in taus:
    train_f = np.mean([scores(prob[i] > tau, dibco.masks[i] == 1)["F"] for i in TRAIN_PAGES])
    test = [scores(prob[i] > tau, dibco.masks[i] == 1) for i in TEST_PAGES]
    print(f"{tau:>5.1f} {train_f:>8.3f} {np.mean([s['F'] for s in test]):>7.3f} "
          f"{np.mean([s['P'] for s in test]):>7.3f} {np.mean([s['R'] for s in test]):>7.3f}")
    if train_f > best_f:
        best_tau, best_f = tau, train_f
print(f"\nbest threshold on the training pages: {best_tau}")

The best threshold on the training pages, 0.7, is not the best on the test pages, where
0.3 wins (F 0.728 against 0.699 at 0.7). On pages it trained on the network is
confident; on new pages its probabilities for real ink are lower. A threshold tuned on
training data inherits that overconfidence — it should be tuned on held-out pages.

## What to carry forward

- **Segmentation answers per pixel.** The output is an image the size of the input, and
  the loss is the classification loss applied at every pixel.
- **Down for context, up for size, skips for detail.** Stride-2 convolutions widen what
  each feature sees; transposed convolutions restore the resolution; skip connections
  bring back the exact positions the way down averaged away.
- **Fully convolutional = any size.** No layer depends on the image size, so a network
  trained on $64 \times 64$ crops segments whole pages.
- **With a rare class, accuracy lies.** Score the class you care about — precision, recall,
  F-measure — and look at *where* the errors are: on edges or on whole strokes.
- **Hold out whole pages**, not crops: crops of one page share its writer and its stains.

## Try it

1. Cut the skip connections: in `forward`, replace `a2` and `a1` inside the two `cpu.cat`
   calls by zeros of the same shape (`cpu.Tensor(np.zeros(a2.shape, np.float32),
   requires_grad=False)`). The shapes still fit; the fine detail no longer arrives. What
   happens to the errors on the edges of strokes?
2. Train without the per-page standardisation (`prepare` returning just `dibco.grey(i)`).
   Which test page suffers most, and why that one?
3. The loss treats a missed ink pixel and a false one the same, but ink is ten times
   rarer. Weight the ink terms by 3 in `pixel_bce`. How do precision and recall move?

### References

- I. Pratikakis, K. Zagoris, P. Kaddas, B. Gatos. *ICFHR 2018 Competition on Handwritten
  Document Image Binarization (H-DIBCO 2018).* ICFHR 2018.
- O. Ronneberger, P. Fischer, T. Brox. *U-Net: Convolutional Networks for Biomedical
  Image Segmentation.* MICCAI 2015. [arXiv:1505.04597](https://arxiv.org/abs/1505.04597)
- J. Long, E. Shelhamer, T. Darrell. *Fully Convolutional Networks for Semantic
  Segmentation.* CVPR 2015. [arXiv:1411.4038](https://arxiv.org/abs/1411.4038)
- V. Dumoulin, F. Visin. *A guide to convolution arithmetic for deep learning.* 2016.
  [arXiv:1603.07285](https://arxiv.org/abs/1603.07285)